# External validation multiseed analysis

This notebook documents the post-training analysis workflow for the dermatology external-validation study.

It starts from completed outputs and prediction files. It does not retrain models. It shows how the final tables, subgroup comparisons, FDR correction, and figures are produced from the saved result files.

Main question: whether internal HAM10000 validation performance is predictively representative of external BOSQUE and subgroup-specific performance claims.

## 1. Setup

In [ ]:
from pathlib import Path
import subprocess
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

ROOT = Path("/home/andres/DermAlgoFairness")
TABLES = ROOT / "outputs" / "tables"
PUB_TABLES = ROOT / "outputs" / "publication_tables"
FIGURES = ROOT / "outputs" / "figures"
PREDICTIONS = ROOT / "outputs" / "predictions"

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

for name, path in {
    "ROOT": ROOT,
    "TABLES": TABLES,
    "PUB_TABLES": PUB_TABLES,
    "FIGURES": FIGURES,
    "PREDICTIONS": PREDICTIONS,
}.items():
    print(f"{name}: {path} | exists={path.exists()}")

## 2. Helper functions

`run_host()` runs scripts on the server host.  
`run_docker()` runs scripts inside the existing Docker container, which is useful for scripts requiring the ML environment.

In [ ]:
def run_host(command, cwd=ROOT):
    print("$", " ".join(command))
    result = subprocess.run(command, cwd=cwd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result

def run_docker(command_inside_container):
    docker_cmd = [
        "docker", "exec", "dermalgo-arxiv-revision",
        "bash", "-lc",
        f"cd /workspace/DermAlgoFairness && {command_inside_container}",
    ]
    print("$", " ".join(docker_cmd))
    result = subprocess.run(docker_cmd, cwd=ROOT, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result

## 3. Check available result files

In [ ]:
print("Aggregated model files:")
for p in sorted(TABLES.glob("*_all_seed_metrics.csv")):
    print(" -", p.relative_to(ROOT))

print("\nSubgroup files:")
for p in sorted(TABLES.glob("*_all_seed_subgroup_metrics.csv")):
    print(" -", p.relative_to(ROOT))

print("\nBootstrap comparison files:")
for p in sorted(TABLES.glob("bosque_light_dark_bootstrap_comparison*.csv")):
    print(" -", p.relative_to(ROOT))

print("\nPublication tables:")
for p in sorted(PUB_TABLES.glob("*")):
    print(" -", p.relative_to(ROOT))

print("\nFigures:")
for p in sorted(FIGURES.glob("*")):
    print(" -", p.relative_to(ROOT))

## 4. Aggregate seed-level results by model

This calls `scripts/05_aggregate_results.py` for each model. It starts from existing evaluation outputs and produces model-level summaries across seeds.

In [ ]:
MODELS = ["resnet50", "densenet121", "mobilenetv2", "efficientnetv2b0", "vgg16"]

for model in MODELS:
    run_host(["python3", "scripts/05_aggregate_results.py", "--model", model])

## 5. Build cross-model summary tables

In [ ]:
metrics = ["accuracy", "precision", "recall", "specificity", "f1", "auc_roc", "auc_pr"]

rows = []
for model in MODELS:
    df = pd.read_csv(TABLES / f"{model}_all_seed_metrics.csv")
    df.insert(0, "model", model)
    rows.append(df)

all_df = pd.concat(rows, ignore_index=True)
summary = (
    all_df.groupby(["model", "dataset"])[metrics]
    .agg(["mean", "std"])
    .reset_index()
)

all_df.to_csv(TABLES / "all_models_all_seed_metrics.csv", index=False)
summary.to_csv(TABLES / "all_models_all_seed_metrics_summary.csv", index=False)

display(summary)

In [ ]:
rows = []
for model in MODELS:
    df = pd.read_csv(TABLES / f"{model}_all_seed_subgroup_metrics.csv")
    df.insert(0, "model", model)
    rows.append(df)

all_subgroup_df = pd.concat(rows, ignore_index=True)
subgroup_summary = (
    all_subgroup_df.groupby(["model", "subgroup_column", "subgroup"])[metrics]
    .agg(["mean", "std"])
    .reset_index()
)

all_subgroup_df.to_csv(TABLES / "all_models_all_seed_subgroup_metrics.csv", index=False)
subgroup_summary.to_csv(TABLES / "all_models_all_seed_subgroup_metrics_summary.csv", index=False)

display(subgroup_summary)

## 6. Internal HAM10000 vs external BOSQUE performance

In [ ]:
overall = pd.read_csv(TABLES / "all_models_all_seed_metrics.csv")

readable_rows = []
for (model, dataset), g in overall.groupby(["model", "dataset"]):
    row = {"model": model, "dataset": dataset}
    for metric in metrics:
        row[metric] = f"{g[metric].mean():.3f} ± {g[metric].std(ddof=1):.3f}"
    readable_rows.append(row)

overall_readable = pd.DataFrame(readable_rows)
display(overall_readable)

## 7. BOSQUE subgroup performance by skin group

In [ ]:
subgroup = pd.read_csv(TABLES / "all_models_all_seed_subgroup_metrics.csv")

subgroup_readable_rows = []
for (model, subgroup_name), g in subgroup.groupby(["model", "subgroup"]):
    row = {"model": model, "subgroup": subgroup_name, "n": int(g["n"].iloc[0]) if "n" in g.columns else None}
    for metric in metrics:
        row[metric] = f"{g[metric].mean():.3f} ± {g[metric].std(ddof=1):.3f}"
    subgroup_readable_rows.append(row)

subgroup_readable = pd.DataFrame(subgroup_readable_rows)
display(subgroup_readable.sort_values(["model", "subgroup"]))

## 8. Bootstrap light–dark subgroup comparison with FDR correction

This calls `scripts/06_compare_bosque_subgroups.py` inside Docker because it depends on the ML environment.

In [ ]:
# This may take a little time due to bootstrap resampling.
run_docker("python3 scripts/06_compare_bosque_subgroups.py --n-boot 2000 --seed 1")

In [ ]:
gaps = pd.read_csv(TABLES / "bosque_light_dark_bootstrap_comparison_readable.csv")

cols = [
    "model", "metric", "dark", "light", "gap_light_minus_dark",
    "bootstrap_ci_low", "bootstrap_ci_high",
    "p_value_bootstrap", "p_value_fdr_bh", "sig_fdr",
]

display(gaps[cols])

## 9. FDR-significant subgroup gaps

In [ ]:
sig = gaps[gaps["significant_fdr_0_05"]].copy()

display(sig[cols].sort_values(["model", "metric"]))

display(Markdown("### Number of significant results by metric"))
display(sig.groupby("metric").size().sort_values(ascending=False).to_frame("n_significant"))

display(Markdown("### Number of significant results by model"))
display(sig.groupby("model").size().sort_values(ascending=False).to_frame("n_significant"))

display(Markdown("### FDR-significant light > dark gaps"))
display(
    sig[sig["gap_light_minus_dark"] > 0]
    [["model", "metric", "gap_light_minus_dark", "p_value_fdr_bh", "sig_fdr"]]
    .sort_values(["model", "metric"])
)

display(Markdown("### FDR-significant dark > light gaps"))
display(
    sig[sig["gap_light_minus_dark"] < 0]
    [["model", "metric", "gap_light_minus_dark", "p_value_fdr_bh", "sig_fdr"]]
    .sort_values(["model", "metric"])
)

## 10. Publication-ready tables

In [ ]:
run_host(["python3", "scripts/07_make_publication_tables.py"])

table1 = pd.read_csv(PUB_TABLES / "table_01_internal_external_performance.csv")
table2 = pd.read_csv(PUB_TABLES / "table_02_bosque_subgroup_performance.csv")
table3 = pd.read_csv(PUB_TABLES / "table_03_fdr_significant_light_dark_gaps.csv")

display(Markdown("### Table 1 — Internal vs external performance"))
display(table1)

display(Markdown("### Table 2 — BOSQUE subgroup performance"))
display(table2)

display(Markdown("### Table 3 — FDR-significant light–dark gaps"))
display(table3)

## 11. Figures

In [ ]:
run_host(["python3", "scripts/08_plot_results.py"])

print("Generated figures:")
for p in sorted(FIGURES.glob("*")):
    print(p.relative_to(ROOT))

In [ ]:
for name in [
    "figure_internal_external_f1.png",
    "figure_internal_external_auc_pr.png",
    "figure_bosque_light_dark_gap_heatmap.png",
    "figure_bosque_significant_gap_forest.png",
]:
    path = FIGURES / name
    print(path.relative_to(ROOT))
    display(Image(filename=str(path)))

## 12. Interpretation

After FDR correction, the most stable subgroup gaps are observed for AUC-PR and precision. Both metrics are significantly higher in the light-skin subgroup across all five architectures. F1-score is significantly higher for most architectures, while recall and AUC-ROC gaps are more architecture-dependent.

This supports the conclusion that internal HAM10000 validation performance does not uniformly transport to the external BOSQUE subgroup setting. In Predictive Representativity terms, the internal validation evidence is insufficient for a homogeneous external performance claim across skin-group Objective Reference Points.

## TAC–ETC decision analysis

This section maps each architecture and BOSQUE target condition into the Target Adequacy Criterion and External Transportability Criterion decision space. The y-axis represents mean target performance across five seeds. The x-axis represents the source-to-target difference between HAM10000 internal validation and the BOSQUE target condition. Negative values indicate that the external BOSQUE performance exceeded the internal estimate.

The thresholds are used here as methodological reference values. They should not be interpreted as clinically definitive unless externally justified.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent
TABLES = ROOT / "outputs" / "tables"

aucpr_tac_etc = pd.read_csv(TABLES / "tac_etc_decision_table_auc_pr.csv")
f1_tac_etc = pd.read_csv(TABLES / "tac_etc_decision_table_f1.csv")

display(aucpr_tac_etc)
display(f1_tac_etc)

,model,target_condition,target_performance,source_performance,degradation,tac_pass,etc_pass,decision_region,metric,tau,epsilon
0,DenseNet121,BOSQUE dark,0.799017,0.732600,-0.066417,False,True,transported but inadequate,auc_pr,0.85,0.05
1,EfficientNetV2B0,BOSQUE dark,0.785503,0.751396,-0.034107,False,True,transported but inadequate,auc_pr,0.85,0.05
2,MobileNetV2,BOSQUE dark,0.696845,0.725674,0.028828,False,True,transported but inadequate,auc_pr,0.85,0.05
3,ResNet50,BOSQUE dark,0.756434,0.789192,0.032758,False,True,transported but inadequate,auc_pr,0.85,0.05
4,VGG16,BOSQUE dark,0.767251,0.747262,-0.019989,False,True,transported but inadequate,auc_pr,0.85,0.05
5,DenseNet121,BOSQUE light,0.930346,0.732600,-0.197745,True,True,maintained,auc_pr,0.85,0.05
6,EfficientNetV2B0,BOSQUE light,0.950770,0.751396,-0.199375,True,True,maintained,auc_pr,0.85,0.05
7,MobileNetV2,BOSQUE light,0.931844,0.725674,-0.206171,True,True,maintained,auc_pr,0.85,0.05
8,ResNet50,BOSQUE light,0.944649,0.789192,-0.155457,True,True,maintained,auc_pr,0.85,0.05
9,VGG16,BOSQUE light,0.920124,0.747262,-0.172862,True,True,maintained,auc_pr,0.85,0.05


,model,target_condition,target_performance,source_performance,degradation,tac_pass,etc_pass,decision_region,metric,tau,epsilon
0,DenseNet121,BOSQUE dark,0.628922,0.670075,0.041154,False,True,transported but inadequate,f1,0.7,0.05
1,EfficientNetV2B0,BOSQUE dark,0.659507,0.685323,0.025815,False,True,transported but inadequate,f1,0.7,0.05
2,MobileNetV2,BOSQUE dark,0.573377,0.671803,0.098426,False,False,restricted / rejected,f1,0.7,0.05
3,ResNet50,BOSQUE dark,0.539617,0.721155,0.181538,False,False,restricted / rejected,f1,0.7,0.05
4,VGG16,BOSQUE dark,0.608260,0.677212,0.068951,False,False,restricted / rejected,f1,0.7,0.05
5,DenseNet121,BOSQUE light,0.737330,0.670075,-0.067254,True,True,maintained,f1,0.7,0.05
6,EfficientNetV2B0,BOSQUE light,0.807063,0.685323,-0.121741,True,True,maintained,f1,0.7,0.05
7,MobileNetV2,BOSQUE light,0.702598,0.671803,-0.030795,True,True,maintained,f1,0.7,0.05
8,ResNet50,BOSQUE light,0.718228,0.721155,0.002926,True,True,maintained,f1,0.7,0.05
9,VGG16,BOSQUE light,0.680506,0.677212,-0.003294,False,True,transported but inadequate,f1,0.7,0.05


## 13. Limitations and next analyses

Main limitations:

1. BOSQUE public test set is small.
2. Dark subgroup has 46 images.
3. Light subgroup has 105 images.
4. Classification metrics are reported at threshold 0.5.
5. Ranking metrics such as AUC-ROC and AUC-PR should be emphasized because they are threshold-independent.
6. Threshold sensitivity and calibration should be analyzed next.
7. Dataset case-mix may differ by subgroup and should be described when possible.